# IPL Database – Complex SQL Query Building

**Goal:** practice building complex SQL queries (joins, CTEs, subqueries, aggregations and window functions) on the Indian Premier League (IPL) database, ending with one query that returns full career metrics per player.

**Contents**
1. Load and explore the data
2. Query 1 – Select all columns from `Player_Match`
3. Query 2 – Batsman vs runs
4. Query 3 – Fifties and hundreds
5. Query 4 – Best bowling figures
6. Query 5 – Comprehensive career metrics

> Run the cells in order (Runtime → Run all).

## 1. Load and explore the data

In [ ]:
import os
import re
import glob
import sqlite3
import zipfile
import urllib.request

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
# Download and unzip the dataset provided for the project
URL = ("https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/"
       "Week%2010%20-%20Advanced%20SQL/W10D4%20-%20Mini%20project/"
       "Approach%20to%20Complex%20SQLquery%20Building%20in%20Kaggle.zip")
ZIP_PATH = "ipl_dataset.zip"
DATA_DIR = "ipl_data"

if not os.path.exists(ZIP_PATH):
    urllib.request.urlretrieve(URL, ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

for root, _, files in os.walk(DATA_DIR):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Connect to the IPL database.
# If the zip contains a SQLite file we open it directly; if it only contains CSV files
# we load them into an in-memory SQLite database (one table per CSV).
db_files = [p for ext in ("*.sqlite", "*.sqlite3", "*.db")
            for p in glob.glob(os.path.join(DATA_DIR, "**", ext), recursive=True)]

if db_files:
    DB_PATH = max(db_files, key=os.path.getsize)
    con = sqlite3.connect(DB_PATH)
    print("Connected to SQLite file:", DB_PATH)
else:
    con = sqlite3.connect(":memory:")
    csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
    for path in csv_files:
        table = os.path.splitext(os.path.basename(path))[0].replace(" ", "_")
        pd.read_csv(path).to_sql(table, con, index=False, if_exists="replace")
    print(f"No SQLite file found. Loaded {len(csv_files)} CSV files into an in-memory database.")


def run_query(sql, params=None):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, con, params=params)

In [ ]:
# The "master table" of SQLite (sqlite_master) describes every object in the database
master = run_query("SELECT type, name, tbl_name FROM sqlite_master WHERE type = 'table' ORDER BY name")
master

In [ ]:
# Print the column names of every table
columns_by_table = {}
for table in master["name"]:
    info = run_query(f"PRAGMA table_info({table})")
    columns_by_table[table] = info["name"].tolist()
    n_rows = run_query(f"SELECT COUNT(*) AS n FROM {table}")["n"][0]
    print(f"{table} ({n_rows:,} rows): {columns_by_table[table]}")

In [ ]:
# Identify the common columns (the ones we can use to JOIN tables)
col_to_tables = {}
for table, cols in columns_by_table.items():
    for c in cols:
        col_to_tables.setdefault(c, []).append(table)

common_columns = {c: t for c, t in col_to_tables.items() if len(t) > 1}
for c, t in sorted(common_columns.items()):
    print(f"{c:20s} -> {t}")

In [ ]:
# Check that the columns used in this notebook exist.
# If something is reported missing, adjust the column name in the queries below.
expected = {
    "Player": ["Player_Id", "Player_Name"],
    "Player_Match": ["Match_Id", "Player_Id"],
    "Ball_by_Ball": ["Match_Id", "Over_Id", "Ball_Id", "Innings_No", "Striker", "Bowler"],
    "Batsman_Scored": ["Match_Id", "Over_Id", "Ball_Id", "Innings_No", "Runs_Scored"],
    "Wicket_Taken": ["Match_Id", "Over_Id", "Ball_Id", "Innings_No", "Player_Out"],
    "Extra_Runs": ["Match_Id", "Over_Id", "Ball_Id", "Innings_No", "Extra_Runs"],
}
lower_cols = {t.lower(): [c.lower() for c in cols] for t, cols in columns_by_table.items()}

all_ok = True
for table, cols in expected.items():
    if table.lower() not in lower_cols:
        print(f"MISSING TABLE: {table}")
        all_ok = False
        continue
    for c in cols:
        if c.lower() not in lower_cols[table.lower()]:
            print(f"MISSING COLUMN: {table}.{c}")
            all_ok = False
print("Schema check passed." if all_ok else "Please review the items above.")

In [ ]:
# Data-quality check: the ball-level key should identify exactly one delivery
key = "Match_Id, Innings_No, Over_Id, Ball_Id"
for table in ["Ball_by_Ball", "Batsman_Scored"]:
    dup = run_query(f"""
        SELECT COUNT(*) AS duplicated_keys FROM (
            SELECT {key} FROM {table} GROUP BY {key} HAVING COUNT(*) > 1
        )
    """)["duplicated_keys"][0]
    print(f"{table}: {dup} duplicated ball keys")

## 2. Query 1 – Select all columns from the `Player_Match` table

In [ ]:
# Query 1
query_1 = """
SELECT *
FROM Player_Match
LIMIT 10
"""
run_query(query_1)

## 3. Query 2 – Batsman vs runs
Every delivery is in `Ball_by_Ball` (who faced it) and the runs of that delivery are in `Batsman_Scored`.
A delivery is identified by `Match_Id + Innings_No + Over_Id + Ball_Id`, so we join on those four columns.
> Super-over innings are included. Extras (wides, byes...) are not credited to the batsman.

In [ ]:
# Query 2
query_2 = """
SELECT
    p.Player_Id,
    p.Player_Name,
    SUM(bs.Runs_Scored) AS total_runs
FROM Ball_by_Ball b
JOIN Batsman_Scored bs
     ON  bs.Match_Id   = b.Match_Id
     AND bs.Innings_No = b.Innings_No
     AND bs.Over_Id    = b.Over_Id
     AND bs.Ball_Id    = b.Ball_Id
JOIN Player p ON p.Player_Id = b.Striker
GROUP BY p.Player_Id, p.Player_Name
ORDER BY total_runs DESC
"""
df_q2 = run_query(query_2)
display(df_q2.head(10))

df_q2.head(10).plot(kind="barh", x="Player_Name", y="total_runs", figsize=(8, 4),
                    legend=False, title="Top 10 run scorers")
plt.gca().invert_yaxis()
plt.xlabel("Total runs")
plt.show()

## 4. Query 3 – Fifties and hundreds
Step 1 (CTE): runs scored by every batsman **in every match**.
Step 2: count the matches with 50–99 runs (fifties) and 100+ runs (hundreds).

In [ ]:
# Query 3
query_3 = """
WITH runs_per_match AS (
    SELECT
        b.Striker   AS player_id,
        b.Match_Id  AS match_id,
        SUM(bs.Runs_Scored) AS match_runs
    FROM Ball_by_Ball b
    JOIN Batsman_Scored bs
         ON  bs.Match_Id   = b.Match_Id
         AND bs.Innings_No = b.Innings_No
         AND bs.Over_Id    = b.Over_Id
         AND bs.Ball_Id    = b.Ball_Id
    GROUP BY b.Striker, b.Match_Id
)
SELECT
    p.Player_Id,
    p.Player_Name,
    COUNT(*)                                                       AS innings,
    SUM(r.match_runs)                                              AS total_runs,
    MAX(r.match_runs)                                              AS highest_score,
    SUM(CASE WHEN r.match_runs >= 50 AND r.match_runs < 100 THEN 1 ELSE 0 END) AS fifties,
    SUM(CASE WHEN r.match_runs >= 100 THEN 1 ELSE 0 END)           AS hundreds
FROM runs_per_match r
JOIN Player p ON p.Player_Id = r.player_id
GROUP BY p.Player_Id, p.Player_Name
HAVING fifties > 0 OR hundreds > 0
ORDER BY hundreds DESC, fifties DESC, total_runs DESC
"""
df_q3 = run_query(query_3)
df_q3.head(15)

## 5. Query 4 – Best bowling figures
Before writing the query we need to decide two rules (cricket conventions):

1. **Wickets credited to the bowler:** every dismissal except *run out*, *retired hurt* and *obstructing the field*.
2. **Runs conceded by the bowler:** runs off the bat + **wides and no-balls**. Byes and leg-byes are *not* charged to the bowler.

Depending on the database version, the dismissal kind and the extra type are stored as text (`'run out'`)
or as ids pointing to a lookup table. The helper below builds the right SQL filter in both cases.

In [ ]:
def norm(s):
    """Normalize a label: lower case, no spaces/underscores (e.g. 'Run Out' -> 'runout')."""
    return re.sub(r"[\s_]+", "", str(s).lower())


def table_columns(table):
    return [r[1] for r in con.execute(f"PRAGMA table_info({table})")]


def find_col(table, candidates):
    """Return the real column name (case-insensitive) from a list of candidates."""
    existing = {c.lower(): c for c in table_columns(table)}
    for cand in candidates:
        if cand.lower() in existing:
            return existing[cand.lower()]
    raise ValueError(f"None of {candidates} found in {table}. Columns: {list(existing.values())}")


def in_filter(table, col, values, lookup_table=None, negate=False):
    """
    Build a SQL condition on `col` that matches the given labels.
    - text column  -> normalized comparison against the labels
    - integer ids  -> labels are translated to ids through `lookup_table` (id, name)
    """
    wanted = {norm(v) for v in values}
    op = "NOT IN" if negate else "IN"
    sample = con.execute(f"SELECT typeof({col}) FROM {table} WHERE {col} IS NOT NULL LIMIT 1").fetchone()
    if sample is None or sample[0] == "text":
        labels = ", ".join(f"'{w}'" for w in sorted(wanted))
        return f"REPLACE(REPLACE(LOWER({col}), ' ', ''), '_', '') {op} ({labels})"
    lcols = table_columns(lookup_table)
    rows = con.execute(f"SELECT {lcols[0]}, {lcols[1]} FROM {lookup_table}").fetchall()
    ids = [str(r[0]) for r in rows if norm(r[1]) in wanted] or ["-1"]
    return f"{col} {op} ({', '.join(ids)})"


# Column that stores the dismissal kind and the extra type
KIND_COL = find_col("Wicket_Taken", ["Kind_Out", "Out_Type", "Out_Id"])
EXTRA_COL = find_col("Extra_Runs", ["Extra_Type", "Extra_Type_Id", "Extra_Id"])
print("Dismissal column:", KIND_COL, "| Extra type column:", EXTRA_COL)

# Filters (used inside the SQL below)
NOT_BOWLER_WICKET = in_filter("Wicket_Taken", KIND_COL,
                              ["run out", "retired hurt", "obstructing the field", "retired out"],
                              lookup_table="Out_Type", negate=True)
IS_DISMISSAL = in_filter("Wicket_Taken", KIND_COL, ["retired hurt"],
                         lookup_table="Out_Type", negate=True)
IS_WIDE = in_filter("Extra_Runs", EXTRA_COL, ["wides", "wide"], lookup_table="Extra_Type")
IS_NOBALL = in_filter("Extra_Runs", EXTRA_COL, ["noballs", "noball"], lookup_table="Extra_Type")

print(NOT_BOWLER_WICKET)
print(IS_DISMISSAL)
print(IS_WIDE)
print(IS_NOBALL)

# Sanity check: what values do these columns contain?
display(run_query(f"SELECT {KIND_COL} AS dismissal, COUNT(*) AS n FROM Wicket_Taken GROUP BY 1 ORDER BY n DESC"))
display(run_query(f"SELECT {EXTRA_COL} AS extra_type, COUNT(*) AS n FROM Extra_Runs GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# Reusable CTE blocks (written once, used in Queries 4 and 5)

# One row per delivery with: batsman runs, whether it was a wide / no-ball and the extra runs
BALL_CTES = f"""
extras AS (
    SELECT
        Match_Id, Innings_No, Over_Id, Ball_Id,
        SUM(CASE WHEN {IS_WIDE}   THEN Extra_Runs ELSE 0 END) AS wide_runs,
        SUM(CASE WHEN {IS_NOBALL} THEN Extra_Runs ELSE 0 END) AS noball_runs,
        MAX(CASE WHEN {IS_WIDE}   THEN 1 ELSE 0 END)          AS is_wide,
        MAX(CASE WHEN {IS_NOBALL} THEN 1 ELSE 0 END)          AS is_noball
    FROM Extra_Runs
    GROUP BY Match_Id, Innings_No, Over_Id, Ball_Id
),
balls AS (
    SELECT
        b.Match_Id, b.Innings_No, b.Over_Id, b.Ball_Id,
        b.Striker, b.Bowler,
        COALESCE(bs.Runs_Scored, 0) AS bat_runs,
        COALESCE(e.wide_runs, 0)    AS wide_runs,
        COALESCE(e.noball_runs, 0)  AS noball_runs,
        COALESCE(e.is_wide, 0)      AS is_wide,
        COALESCE(e.is_noball, 0)    AS is_noball
    FROM Ball_by_Ball b
    LEFT JOIN Batsman_Scored bs
           ON  bs.Match_Id = b.Match_Id AND bs.Innings_No = b.Innings_No
           AND bs.Over_Id  = b.Over_Id  AND bs.Ball_Id    = b.Ball_Id
    LEFT JOIN extras e
           ON  e.Match_Id = b.Match_Id AND e.Innings_No = b.Innings_No
           AND e.Over_Id  = b.Over_Id  AND e.Ball_Id    = b.Ball_Id
),
bowler_wickets AS (
    SELECT b.Bowler AS bowler_id, b.Match_Id AS match_id, COUNT(*) AS wickets
    FROM Wicket_Taken w
    JOIN Ball_by_Ball b
         ON  b.Match_Id = w.Match_Id AND b.Innings_No = w.Innings_No
         AND b.Over_Id  = w.Over_Id  AND b.Ball_Id    = w.Ball_Id
    WHERE {NOT_BOWLER_WICKET}
    GROUP BY b.Bowler, b.Match_Id
),
bowling_per_match AS (
    SELECT
        ba.Bowler   AS bowler_id,
        ba.Match_Id AS match_id,
        SUM(ba.bat_runs + ba.wide_runs + ba.noball_runs)   AS runs_conceded,
        SUM(1 - ba.is_wide - ba.is_noball)                 AS legal_balls,
        COALESCE(MAX(bw.wickets), 0)                       AS wickets
    FROM balls ba
    LEFT JOIN bowler_wickets bw
           ON bw.bowler_id = ba.Bowler AND bw.match_id = ba.Match_Id
    GROUP BY ba.Bowler, ba.Match_Id
),
best_bowling AS (
    SELECT
        bowler_id, match_id, wickets, runs_conceded,
        ROW_NUMBER() OVER (
            PARTITION BY bowler_id
            ORDER BY wickets DESC, runs_conceded ASC
        ) AS rn
    FROM bowling_per_match
)
"""

# Query 4
query_4 = f"""
WITH {BALL_CTES}
SELECT
    p.Player_Id,
    p.Player_Name,
    bb.wickets || '/' || bb.runs_conceded AS best_bowling_figures,
    bb.wickets       AS best_wickets,
    bb.runs_conceded AS best_runs_conceded,
    bb.match_id      AS match_id
FROM best_bowling bb
JOIN Player p ON p.Player_Id = bb.bowler_id
WHERE bb.rn = 1
ORDER BY bb.wickets DESC, bb.runs_conceded ASC
"""
df_q4 = run_query(query_4)
df_q4.head(15)

## 6. Query 5 – Comprehensive career metrics
We combine everything into one query. Building blocks (CTEs):

| CTE | What it does |
|---|---|
| `batting_per_match` | runs, balls faced, fours and sixes of every batsman in every match |
| `batting_career` | innings, runs, highest score, fifties, hundreds, strike rate |
| `dismissals` | times each player was dismissed → batting average |
| `bowling_per_match` / `best_bowling` | wickets and runs per match → best figures (Query 4) |
| `bowling_career` | wickets, runs conceded, overs, economy rate |
| `matches_played` | matches per player from `Player_Match` |

The final `SELECT` joins them all to `Player` with `LEFT JOIN`, so players who only bat or only bowl are kept.

In [ ]:
# Query 5
query_5 = f"""
WITH {BALL_CTES},
batting_per_match AS (
    SELECT
        Striker  AS player_id,
        Match_Id AS match_id,
        SUM(bat_runs)                              AS runs,
        SUM(1 - is_wide)                           AS balls_faced,
        SUM(CASE WHEN bat_runs = 4 THEN 1 ELSE 0 END) AS fours,
        SUM(CASE WHEN bat_runs = 6 THEN 1 ELSE 0 END) AS sixes
    FROM balls
    GROUP BY Striker, Match_Id
),
batting_career AS (
    SELECT
        player_id,
        COUNT(*)                                   AS innings,
        SUM(runs)                                  AS total_runs,
        SUM(balls_faced)                           AS balls_faced,
        MAX(runs)                                  AS highest_score,
        SUM(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS fifties,
        SUM(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS hundreds,
        SUM(fours)                                 AS fours,
        SUM(sixes)                                 AS sixes
    FROM batting_per_match
    GROUP BY player_id
),
dismissals AS (
    SELECT Player_Out AS player_id, COUNT(*) AS times_out
    FROM Wicket_Taken
    WHERE {IS_DISMISSAL}
    GROUP BY Player_Out
),
bowling_career AS (
    SELECT
        bowler_id                AS player_id,
        COUNT(*)                 AS matches_bowled,
        SUM(wickets)             AS wickets,
        SUM(runs_conceded)       AS runs_conceded,
        SUM(legal_balls)         AS balls_bowled
    FROM bowling_per_match
    GROUP BY bowler_id
),
matches_played AS (
    SELECT Player_Id AS player_id, COUNT(DISTINCT Match_Id) AS matches
    FROM Player_Match
    GROUP BY Player_Id
)
SELECT
    p.Player_Id,
    p.Player_Name,
    COALESCE(mp.matches, 0)                          AS matches,
    -- Batting
    COALESCE(bc.innings, 0)                          AS batting_innings,
    COALESCE(bc.total_runs, 0)                       AS total_runs,
    COALESCE(bc.highest_score, 0)                    AS highest_score,
    ROUND(1.0 * bc.total_runs / NULLIF(d.times_out, 0), 2)    AS batting_average,
    ROUND(100.0 * bc.total_runs / NULLIF(bc.balls_faced, 0), 2) AS strike_rate,
    COALESCE(bc.fifties, 0)                          AS fifties,
    COALESCE(bc.hundreds, 0)                         AS hundreds,
    COALESCE(bc.fours, 0)                            AS fours,
    COALESCE(bc.sixes, 0)                            AS sixes,
    -- Bowling
    COALESCE(bw.matches_bowled, 0)                   AS bowling_matches,
    COALESCE(bw.wickets, 0)                          AS wickets,
    COALESCE(bw.runs_conceded, 0)                    AS runs_conceded,
    ROUND(6.0 * bw.runs_conceded / NULLIF(bw.balls_bowled, 0), 2) AS economy_rate,
    ROUND(1.0 * bw.runs_conceded / NULLIF(bw.wickets, 0), 2)      AS bowling_average,
    CASE WHEN bb.bowler_id IS NULL THEN NULL
         ELSE bb.wickets || '/' || bb.runs_conceded END           AS best_bowling_figures
FROM Player p
LEFT JOIN matches_played mp ON mp.player_id = p.Player_Id
LEFT JOIN batting_career bc ON bc.player_id = p.Player_Id
LEFT JOIN dismissals d      ON d.player_id  = p.Player_Id
LEFT JOIN bowling_career bw ON bw.player_id = p.Player_Id
LEFT JOIN best_bowling bb   ON bb.bowler_id = p.Player_Id AND bb.rn = 1
WHERE bc.player_id IS NOT NULL OR bw.player_id IS NOT NULL
ORDER BY total_runs DESC
"""
df_q5 = run_query(query_5)
print(f"{len(df_q5)} players")
df_q5.head(20)

In [ ]:
# Quick looks at the final table
print("Top 10 by wickets")
display(df_q5.sort_values("wickets", ascending=False).head(10)[
    ["Player_Name", "matches", "wickets", "economy_rate", "bowling_average", "best_bowling_figures"]])

print("Top 10 by sixes")
display(df_q5.sort_values("sixes", ascending=False).head(10)[
    ["Player_Name", "matches", "total_runs", "sixes", "fours", "strike_rate"]])

# Consistency check: Query 5 runs should match Query 2
check = df_q2.merge(df_q5[["Player_Id", "total_runs"]], on="Player_Id", suffixes=("_q2", "_q5"))
print("Runs match between Query 2 and Query 5:", (check["total_runs_q2"] == check["total_runs_q5"]).all())

In [ ]:
# Scatter: batting average vs strike rate for players with 500+ runs
subset = df_q5[df_q5["total_runs"] >= 500].dropna(subset=["batting_average", "strike_rate"])
ax = subset.plot(kind="scatter", x="strike_rate", y="batting_average", figsize=(8, 5),
                 title="Batting average vs strike rate (500+ career runs)")
for _, row in subset.nlargest(8, "total_runs").iterrows():
    ax.annotate(row["Player_Name"], (row["strike_rate"], row["batting_average"]), fontsize=8)
plt.show()

## 7. Key insights
Write your conclusions after running the notebook, for example:
- Who are the top run scorers and how do their strike rates compare?
- Which batsmen have the most hundreds and fifties?
- Which bowlers have the best figures in a single match and the best career economy rate?
- Are there all-rounders (many runs *and* many wickets)?